In [ ]:
from google.colab import drive
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)

os.makedirs(
    TABLE_PATH,
    exist_ok=True
)

# Load canonical 10-station dataset
model_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_10_station_feature_engineered.parquet"
    )
)

model_df["Datetime"] = pd.to_datetime(
    model_df["Datetime"]
)

model_df["StationId"] = (
    model_df["StationId"]
    .astype(str)
)

model_df = (
    model_df
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

print(
    "10-station rows:",
    len(model_df)
)

print(
    "Stations:",
    model_df["StationId"].nunique()
)

In [ ]:
RAW_FEATURES = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

TARGET = "AQI_t_plus_1"

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{column}_lag_{lag}h"
    for lag in [1, 2, 12]
    for column in RAW_FEATURES
]

rolling_features = [
    "AQI_rolling_mean_3h",
    "AQI_rolling_std_3h",
    "AQI_rolling_mean_12h",
    "AQI_rolling_std_12h",

    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_std_3h",
    "PM2.5_rolling_mean_12h",
    "PM2.5_rolling_std_12h",

    "PM10_rolling_mean_3h",
    "PM10_rolling_std_3h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_std_12h",

    "NO2_rolling_mean_3h",
    "NO2_rolling_std_3h",
    "NO2_rolling_mean_12h",
    "NO2_rolling_std_12h"
]

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

numeric_features = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
)

model_features = (
    numeric_features
    + ["StationId"]
)

print(
    "Total model features:",
    len(model_features)
)

In [ ]:
unique_times = np.sort(
    model_df["Datetime"].unique()
)

split_position = int(
    len(unique_times) * 0.80
)

train_end_time = unique_times[
    split_position - 1
]

test_10 = (
    model_df[
        model_df["Datetime"] > train_end_time
    ]
    .copy()
    .reset_index(drop=True)
)

X_test_10 = test_10[
    model_features
].copy()

y_test_10 = test_10[
    TARGET
].copy()

print(
    "Test rows:",
    len(test_10)
)

print(
    "Test start:",
    test_10["Datetime"].min()
)

print(
    "Test end:",
    test_10["Datetime"].max()
)

In [ ]:
linear_model_10 = joblib.load(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_linear_regression.joblib"
    )
)

decision_tree_10 = joblib.load(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_decision_tree.joblib"
    )
)

random_forest_10 = joblib.load(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_random_forest.joblib"
    )
)

xgb_10 = joblib.load(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_xgboost.joblib"
    )

)

print("All classical models loaded.")

In [ ]:
predictions_10 = {}

predictions_10["Persistence Baseline"] = (
    test_10["AQI"]
    .to_numpy(
        dtype=float
    )
)

predictions_10["Linear Regression"] = (
    linear_model_10
    .predict(X_test_10)
)

predictions_10["Decision Tree"] = (
    decision_tree_10
    .predict(X_test_10)
)

predictions_10["Random Forest"] = (
    random_forest_10
    .predict(X_test_10)
)

predictions_10["XGBoost"] = (
    xgb_10
    .predict(X_test_10)
)

print(
    "Classical predictions generated."
)

In [ ]:
import tensorflow as tf

lstm_10 = tf.keras.models.load_model(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_lstm.keras"
    )
)

cnn_bilstm_10 = tf.keras.models.load_model(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_cnn_bilstm.keras"
    )
)

dl_imputer = joblib.load(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_dl_imputer.joblib"
    )
)

dl_scaler = joblib.load(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_dl_scaler.joblib"
    )
)

station_encoder = joblib.load(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_encoder.joblib"
    )
)

dl_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Hour",
    "Day",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

all_numeric = dl_imputer.transform(
    model_df[dl_features]
)

all_numeric = dl_scaler.transform(
    all_numeric
)

all_station = station_encoder.transform(
    model_df[["StationId"]]
)

all_processed = np.hstack(
    [
        all_numeric,
        all_station
    ]
).astype(
    "float32"
)

LOOKBACK = 12

station_ids = (
    model_df["StationId"]
    .astype(str)
    .to_numpy()
)

timestamps = (
    model_df["Datetime"]
    .to_numpy()
)

targets = (
    model_df[TARGET]
    .to_numpy(dtype="float32")
)

X_dl_test = []
y_dl_test = []
dl_metadata = []

test_start = test_10["Datetime"].min()

for station_id in np.unique(
    station_ids
):

    positions = np.where(
        station_ids == station_id
    )[0]

    positions = positions[
        np.argsort(
            timestamps[positions]
        )
    ]

    for i in range(
        LOOKBACK,
        len(positions)
    ):

        current_position = positions[i]

        current_time = pd.Timestamp(
            timestamps[current_position]
        )

        if current_time <= test_start:
            continue

        window_positions = positions[
            i - LOOKBACK:i
        ]

        window_times = timestamps[
            window_positions
        ]

        diffs = np.diff(
            window_times.astype(
                "datetime64[s]"
            ).astype(
                np.int64
            )
        )

        if not np.all(
            diffs == 3600
        ):
            continue

        target_value = targets[
            current_position
        ]

        if not np.isfinite(
            target_value
        ):
            continue

        X_dl_test.append(
            all_processed[
                window_positions
            ]
        )

        y_dl_test.append(
            target_value
        )

        dl_metadata.append(
            {
                "StationId": station_id,
                "Datetime": current_time
            }
        )

X_dl_test = np.asarray(
    X_dl_test,
    dtype="float32"
)

y_dl_test = np.asarray(
    y_dl_test,
    dtype="float32"
)

dl_metadata = pd.DataFrame(
    dl_metadata
)

lstm_pred = (
    lstm_10
    .predict(
        X_dl_test,
        batch_size=256,
        verbose=1
    )
    .ravel()
)

cnn_pred = (
    cnn_bilstm_10
    .predict(
        X_dl_test,
        batch_size=256,
        verbose=1
    )
    .ravel()
)

print(
    "LSTM predictions:",
    len(lstm_pred)
)

print(
    "CNN-BiLSTM predictions:",
    len(cnn_pred)
)

In [ ]:
def prediction_accuracy_percent(
    actual,
    predicted,
    tolerance
):
    actual = np.asarray(
        actual,
        dtype=float
    )

    predicted = np.asarray(
        predicted,
        dtype=float
    )

    valid = (
        np.isfinite(actual)
        &
        np.isfinite(predicted)
    )

    actual = actual[valid]
    predicted = predicted[valid]

    absolute_error = np.abs(
        actual - predicted
    )

    return (
        np.mean(
            absolute_error <= tolerance
        )
        * 100
    )


def build_metrics(
    model_name,
    actual,
    predicted
):
    return {
        "Model": model_name,

        "MAE": mean_absolute_error(
            actual,
            predicted
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        ),

        "R2": r2_score(
            actual,
            predicted
        ),

        "PredictionAccuracy_±10AQI_%":
            prediction_accuracy_percent(
                actual,
                predicted,
                10
            ),

        "PredictionAccuracy_±20AQI_%":
            prediction_accuracy_percent(
                actual,
                predicted,
                20
            ),

        "TestRecords": len(actual)
    }

In [ ]:
# ============================================================
# STEP 12.6.7 — FIXED METRIC FUNCTION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


def prediction_accuracy_percent(
    actual,
    predicted,
    tolerance
):
    actual = np.asarray(
        actual,
        dtype=float
    )

    predicted = np.asarray(
        predicted,
        dtype=float
    )

    valid = (
        np.isfinite(actual)
        &
        np.isfinite(predicted)
    )

    actual = actual[valid]
    predicted = predicted[valid]

    if len(actual) == 0:
        return np.nan

    absolute_error = np.abs(
        actual - predicted
    )

    return (
        np.mean(
            absolute_error <= tolerance
        )
        * 100
    )


def build_metrics(
    model_name,
    actual,
    predicted
):
    actual = np.asarray(
        actual,
        dtype=float
    )

    predicted = np.asarray(
        predicted,
        dtype=float
    )

    # --------------------------------------------------------
    # Make sure lengths match
    # --------------------------------------------------------

    if len(actual) != len(predicted):

        raise ValueError(
            f"{model_name}: "
            f"actual length = {len(actual)}, "
            f"prediction length = {len(predicted)}"
        )

    # --------------------------------------------------------
    # Remove NaN / inf pairs
    # --------------------------------------------------------

    valid = (
        np.isfinite(actual)
        &
        np.isfinite(predicted)
    )

    valid_actual = actual[valid]
    valid_predicted = predicted[valid]

    excluded = (
        len(actual)
        - len(valid_actual)
    )

    if len(valid_actual) == 0:

        raise ValueError(
            f"{model_name}: no valid "
            "actual/prediction pairs."
        )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    return {
        "Model": model_name,

        "MAE": mean_absolute_error(
            valid_actual,
            valid_predicted
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                valid_actual,
                valid_predicted
            )
        ),

        "R2": r2_score(
            valid_actual,
            valid_predicted
        ),

        "PredictionAccuracy_±10AQI_%":
            prediction_accuracy_percent(
                valid_actual,
                valid_predicted,
                10
            ),

        "PredictionAccuracy_±20AQI_%":
            prediction_accuracy_percent(
                valid_actual,
                valid_predicted,
                20
            ),

        "ValidTestRecords": len(
            valid_actual
        ),

        "ExcludedInvalidRecords": excluded
    }

print(
    "Fixed metric function ready."
)

In [ ]:
# ============================================================
# STEP 12.6.8 — FIXED FINAL 10-STATION ACCURACY
# ============================================================

accuracy_rows_10 = []

# ------------------------------------------------------------
# Classical models
# ------------------------------------------------------------

for model_name, prediction in predictions_10.items():

    accuracy_rows_10.append(
        build_metrics(
            model_name,
            y_test_10.to_numpy(),
            prediction
        )
    )


# ------------------------------------------------------------
# Deep-learning models
# ------------------------------------------------------------

accuracy_rows_10.append(
    build_metrics(
        "LSTM",
        y_dl_test,
        lstm_pred
    )
)

accuracy_rows_10.append(
    build_metrics(
        "CNN-BiLSTM",
        y_dl_test,
        cnn_pred
    )
)


# ------------------------------------------------------------
# Create final table
# ------------------------------------------------------------

accuracy_results_10 = pd.DataFrame(
    accuracy_rows_10
)

accuracy_results_10 = (
    accuracy_results_10
    .sort_values(
        "RMSE"
    )
    .reset_index(drop=True)
)

# Round presentation values
accuracy_results_10[
    "MAE"
] = accuracy_results_10[
    "MAE"
].round(4)

accuracy_results_10[
    "RMSE"
] = accuracy_results_10[
    "RMSE"
].round(4)

accuracy_results_10[
    "R2"
] = accuracy_results_10[
    "R2"
].round(6)

accuracy_results_10[
    "PredictionAccuracy_±10AQI_%"
] = (
    accuracy_results_10[
        "PredictionAccuracy_±10AQI_%"
    ].round(2)
)

accuracy_results_10[
    "PredictionAccuracy_±20AQI_%"
] = (
    accuracy_results_10[
        "PredictionAccuracy_±20AQI_%"
    ].round(2)
)

print("=" * 110)
print(
    "FINAL 10-STATION — ACTUAL TEST PREDICTION ACCURACY"
)
print("=" * 110)

display(
    accuracy_results_10
)

In [ ]:
# ============================================================
# STEP 12.6.8A — NAN DIAGNOSTIC
# ============================================================

print("=" * 90)
print("PREDICTION NAN / INF DIAGNOSTIC")
print("=" * 90)

for model_name, prediction in predictions_10.items():

    prediction_array = np.asarray(
        prediction,
        dtype=float
    )

    print(
        f"{model_name:25s}"
        f" length={len(prediction_array):8d}"
        f" NaN={np.isnan(prediction_array).sum():6d}"
        f" Inf={np.isinf(prediction_array).sum():6d}"
    )

print(
    f"{'LSTM':25s}"
    f" length={len(lstm_pred):8d}"
    f" NaN={np.isnan(lstm_pred).sum():6d}"
    f" Inf={np.isinf(lstm_pred).sum():6d}"
)

print(
    f"{'CNN-BiLSTM':25s}"
    f" length={len(cnn_pred):8d}"
    f" NaN={np.isnan(cnn_pred).sum():6d}"
    f" Inf={np.isinf(cnn_pred).sum():6d}"
)

print(
    "\nActual classical test targets:"
)

print(
    "NaN:",
    np.isnan(
        y_test_10.to_numpy(
            dtype=float
        )
    ).sum()
)

print(
    "Inf:",
    np.isinf(
        y_test_10.to_numpy(
            dtype=float
        )
    ).sum()
)

print(
    "\nActual deep-learning test targets:"
)

print(
    "NaN:",
    np.isnan(
        y_dl_test
    ).sum()
)

print(
    "Inf:",
    np.isinf(
        y_dl_test
    ).sum()
)

In [ ]:
# ============================================================
# VMD-LSTM RESEARCH ACCURACY
# ============================================================

import glob

vmd_model_candidates = glob.glob(
    os.path.join(
        MODEL_PATH,
        "*vmd*lstm*.keras"
    )
)

print(
    "Saved VMD-LSTM candidates:"
)

for path in vmd_model_candidates:
    print(path)

In [ ]:
vmd_result_path = os.path.join(
    TABLE_PATH,
    "phase9_vmd_results.csv"
)

if os.path.exists(
    vmd_result_path
):

    vmd_results = pd.read_csv(
        vmd_result_path
    )

    print(
        "Saved VMD-LSTM metrics:"
    )

    display(
        vmd_results
    )

else:

    print(
        "Phase 9 VMD result file not found."
    )

In [ ]:
# ============================================================
# TIMEGAN TEST RECONSTRUCTION
# ============================================================

timegan_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO2",
    "NOx",
    "CO",
    "SO2",
    "O3"
]

timegan_station_id = "DL031"

phase5_raw = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase5_feature_engineered.parquet"
    )
)

phase5_raw["Datetime"] = pd.to_datetime(
    phase5_raw["Datetime"]
)

phase5_raw["StationId"] = (
    phase5_raw["StationId"]
    .astype(str)
)

station_data = (
    phase5_raw[
        phase5_raw["StationId"]
        == timegan_station_id
    ]
    .sort_values("Datetime")
    .copy()
)

station_data = (
    station_data[
        ["Datetime"] + timegan_features
    ]
    .dropna(
        subset=timegan_features
    )
    .reset_index(drop=True)
)

print(
    "TimeGAN station rows:",
    len(station_data)
)

In [ ]:
# ============================================================
# TIMEGAN TEST SEQUENCES
# ============================================================

timegan_scaler_path = os.path.join(
    MODEL_PATH,
    "phase10_timegan_scaler.joblib"
)

timegan_scaler = joblib.load(
    timegan_scaler_path
)

scaled_values = (
    timegan_scaler.transform(
        station_data[
            timegan_features
        ]
    )
)

timegan_unique_times = np.sort(
    station_data["Datetime"].unique()
)

timegan_split = int(
    len(timegan_unique_times) * 0.80
)

timegan_test_start = (
    timegan_unique_times[
        timegan_split
    ]
)

aqi_index = timegan_features.index(
    "AQI"
)

X_timegan_test = []
y_timegan_test_scaled = []
timegan_test_times = []

values = scaled_values.astype(
    "float32"
)

times = (
    station_data["Datetime"]
    .reset_index(drop=True)
)

for i in range(
    12,
    len(values)
):

    current_time = times.iloc[i]

    if current_time <= timegan_test_start:
        continue

    window_times = times.iloc[
        i - 12:i + 1
    ]

    if not (
        window_times.diff()
        .dropna()
        .eq(
            pd.Timedelta(hours=1)
        )
        .all()
    ):
        continue

    X_timegan_test.append(
        values[i - 12:i]
    )

    y_timegan_test_scaled.append(
        values[
            i,
            aqi_index
        ]
    )

    timegan_test_times.append(
        current_time
    )

X_timegan_test = np.asarray(
    X_timegan_test,
    dtype="float32"
)

y_timegan_test_scaled = np.asarray(
    y_timegan_test_scaled,
    dtype="float32"
)

# Convert scaled AQI back to original scale.
aqi_min = (
    timegan_scaler
    .data_min_[
        aqi_index
    ]
)

aqi_max = (
    timegan_scaler
    .data_max_[
        aqi_index
    ]
)

def inverse_aqi(
    x
):
    return (
        (
            x + 1
        ) / 2
        * (
            aqi_max - aqi_min
        )
        + aqi_min
    )

y_timegan_test = inverse_aqi(
    y_timegan_test_scaled
)

print(
    "TimeGAN test sequences:",
    X_timegan_test.shape
)

In [ ]:
import tensorflow as tf

real_only_timegan_lstm = tf.keras.models.load_model(
    os.path.join(
        MODEL_PATH,
        "phase10_real_only_lstm.keras"
    )
)

augmented_timegan_lstm = tf.keras.models.load_model(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_augmented_lstm.keras"
    )
)

print(
    "TimeGAN forecasting models loaded."
)

In [ ]:
real_timegan_pred_scaled = (
    real_only_timegan_lstm
    .predict(
        X_timegan_test,
        batch_size=256,
        verbose=1
    )
    .ravel()
)

augmented_timegan_pred_scaled = (
    augmented_timegan_lstm
    .predict(
        X_timegan_test,
        batch_size=256,
        verbose=1
    )
    .ravel()
)

real_timegan_pred = inverse_aqi(
    real_timegan_pred_scaled
)

augmented_timegan_pred = inverse_aqi(
    augmented_timegan_pred_scaled
)

timegan_accuracy_rows = [
    build_metrics(
        "Real-only LSTM - TimeGAN Experiment",
        y_timegan_test,
        real_timegan_pred
    ),

    build_metrics(
        "TimeGAN-Augmented LSTM",
        y_timegan_test,
        augmented_timegan_pred
    )
]

timegan_accuracy_results = pd.DataFrame(
    timegan_accuracy_rows
).round(
    {
        "MAE": 4,
        "RMSE": 4,
        "R2": 6,
        "PredictionAccuracy_±10AQI_%": 2,
        "PredictionAccuracy_±20AQI_%": 2
    }
)

display(
    timegan_accuracy_results
)

In [ ]:
# ============================================================
# STEP 12.6.15 — FIXED COMPLETE MODEL ACCURACY TABLE
# ============================================================

complete_accuracy = []

# ------------------------------------------------------------
# 1. FINAL 10-STATION MODELS
# ------------------------------------------------------------

for _, row in accuracy_results_10.iterrows():

    complete_accuracy.append({
        "Model": row["Model"],

        "Category": (
            "Final Production Candidate"
            if row["Model"] == "Random Forest"
            else "Final 10-Station Comparison"
        ),

        "EvaluationScope": (
            "Final 10 Delhi Stations"
        ),

        "MAE": row["MAE"],
        "RMSE": row["RMSE"],
        "R2": row["R2"],

        "PredictionAccuracy_±10AQI_%": (
            row["PredictionAccuracy_±10AQI_%"]
        ),

        "PredictionAccuracy_±20AQI_%": (
            row["PredictionAccuracy_±20AQI_%"]
        ),

        # FIX: new metric function uses ValidTestRecords
        "ValidTestRecords": (
            row["ValidTestRecords"]
        ),

        "ExcludedInvalidRecords": (
            row["ExcludedInvalidRecords"]
        )
    })


# ------------------------------------------------------------
# 2. TIMEGAN MODELS
# ------------------------------------------------------------

for _, row in timegan_accuracy_results.iterrows():

    complete_accuracy.append({
        "Model": row["Model"],

        "Category": "Research / Generative",

        "EvaluationScope": (
            "R K Puram - Phase 10 experiment"
        ),

        "MAE": row["MAE"],
        "RMSE": row["RMSE"],
        "R2": row["R2"],

        "PredictionAccuracy_±10AQI_%": (
            row["PredictionAccuracy_±10AQI_%"]
        ),

        "PredictionAccuracy_±20AQI_%": (
            row["PredictionAccuracy_±20AQI_%"]
        ),

        "ValidTestRecords": (
            row["ValidTestRecords"]
            if "ValidTestRecords" in row.index
            else np.nan
        ),

        "ExcludedInvalidRecords": (
            row["ExcludedInvalidRecords"]
            if "ExcludedInvalidRecords" in row.index
            else np.nan
        )
    })


# ------------------------------------------------------------
# 3. VMD-LSTM
# ------------------------------------------------------------

if (
    "vmd_results" in globals()
    and len(vmd_results) > 0
):

    # Take the VMD-LSTM row if a Model column exists.
    if "Model" in vmd_results.columns:

        vmd_candidates = vmd_results[
            vmd_results["Model"]
            .astype(str)
            .str.contains(
                "VMD",
                case=False,
                na=False
            )
        ]

        if len(vmd_candidates) > 0:
            vmd_row = vmd_candidates.iloc[0]
        else:
            vmd_row = vmd_results.iloc[0]

    else:
        vmd_row = vmd_results.iloc[0]

    complete_accuracy.append({
        "Model": "VMD-LSTM",

        "Category": "Research / Hybrid",

        "EvaluationScope": (
            "5 selected Delhi Stations - Phase 9"
        ),

        "MAE": vmd_row["MAE"],
        "RMSE": vmd_row["RMSE"],
        "R2": vmd_row["R2"],

        # We did not save the original VMD predictions,
        # so these cannot be calculated retrospectively.
        "PredictionAccuracy_±10AQI_%": np.nan,

        "PredictionAccuracy_±20AQI_%": np.nan,

        "ValidTestRecords": np.nan,

        "ExcludedInvalidRecords": np.nan
    })


# ------------------------------------------------------------
# 4. CREATE MASTER TABLE
# ------------------------------------------------------------

complete_accuracy_df = pd.DataFrame(
    complete_accuracy
)

# Convert numeric columns safely
numeric_columns = [
    "MAE",
    "RMSE",
    "R2",
    "PredictionAccuracy_±10AQI_%",
    "PredictionAccuracy_±20AQI_%",
    "ValidTestRecords",
    "ExcludedInvalidRecords"
]

for column in numeric_columns:

    if column in complete_accuracy_df.columns:

        complete_accuracy_df[column] = pd.to_numeric(
            complete_accuracy_df[column],
            errors="coerce"
        )

# Sort models with valid RMSE first
complete_accuracy_df = (
    complete_accuracy_df
    .sort_values(
        "RMSE",
        na_position="last"
    )
    .reset_index(drop=True)
)

# Round display values
for column in [
    "MAE",
    "RMSE",
    "R2",
    "PredictionAccuracy_±10AQI_%",
    "PredictionAccuracy_±20AQI_%"
]:

    if column in complete_accuracy_df.columns:

        complete_accuracy_df[column] = (
            complete_accuracy_df[column]
            .round(4)
        )

print("=" * 120)
print(
    "ALL MODELS — COMPLETE TEST PREDICTION ACCURACY"
)
print("=" * 120)

display(
    complete_accuracy_df
)

In [ ]:
final_accuracy_winner = (
    accuracy_results_10
    .sort_values(
        "PredictionAccuracy_±10AQI_%",
        ascending=False
    )
    .iloc[0]
)

print("=" * 80)
print(
    "BEST FINAL 10-STATION PREDICTION ACCURACY"
)
print("=" * 80)

print(
    "Model:",
    final_accuracy_winner["Model"]
)

print(
    "Prediction Accuracy ±10 AQI:",
    final_accuracy_winner[
        "PredictionAccuracy_±10AQI_%"
    ],
    "%"
)

print(
    "Prediction Accuracy ±20 AQI:",
    final_accuracy_winner[
        "PredictionAccuracy_±20AQI_%"
    ],
    "%"
)

print(
    "MAE:",
    final_accuracy_winner["MAE"]
)

print(
    "RMSE:",
    final_accuracy_winner["RMSE"]
)

print(
    "R²:",
    final_accuracy_winner["R2"]
)

In [ ]:
complete_csv = os.path.join(
    TABLE_PATH,
    "phase12_6_all_models_prediction_accuracy.csv"
)

complete_xlsx = os.path.join(
    TABLE_PATH,
    "phase12_6_all_models_prediction_accuracy.xlsx"
)

complete_accuracy_df.to_csv(
    complete_csv,
    index=False
)

complete_accuracy_df.to_excel(
    complete_xlsx,
    index=False
)

final_accuracy_csv = os.path.join(
    TABLE_PATH,
    "phase12_6_final_10_station_prediction_accuracy.csv"
)

accuracy_results_10.to_csv(
    final_accuracy_csv,
    index=False
)

print(
    "Complete accuracy table:",
    complete_csv
)

print(
    "Excel table:",
    complete_xlsx
)

print(
    "Final 10-station accuracy table:",
    final_accuracy_csv
)

In [ ]:
phase12_6_status = {
    "completed_phase": (
        "Phase 12.6 - Actual Model Prediction Accuracy"
    ),

    "next_phase": (
        "Phase 13A - Live Adapter + Offline Replay"
    ),

    "accuracy_definition": (
        "Percentage of test predictions within "
        "the specified AQI tolerance"
    ),

    "primary_accuracy": (
        "PredictionAccuracy_±10AQI_%"
    ),

    "secondary_accuracy": (
        "PredictionAccuracy_±20AQI_%"
    ),

    "final_10_station_models_evaluated": [
        "Persistence Baseline",
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost",
        "LSTM",
        "CNN-BiLSTM"
    ],

    "research_models_evaluated": [
        "VMD-LSTM",
        "Real-only LSTM - TimeGAN Experiment",
        "TimeGAN-Augmented LSTM"
    ]
}

status_path = os.path.join(
    OUTPUT_PATH,
    "phase12_6_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        phase12_6_status,
        f,
        indent=4
    )

print(
    "Phase 12.6 status saved."
)

In [ ]:
# ============================================================
# STEP 12.6.19 — FINAL DISPLAY
# ============================================================

print("=" * 120)
print(
    "PHASE 12.6 — COMPLETE MODEL PREDICTION ACCURACY"
)
print("=" * 120)

# ------------------------------------------------------------
# Display only columns that actually exist
# ------------------------------------------------------------

display_columns = [
    "Model",
    "Category",
    "EvaluationScope",
    "MAE",
    "RMSE",
    "R2",
    "PredictionAccuracy_±10AQI_%",
    "PredictionAccuracy_±20AQI_%",
    "ValidTestRecords",
    "ExcludedInvalidRecords"
]

display_columns = [
    column
    for column in display_columns
    if column in complete_accuracy_df.columns
]

print(
    "Available columns:"
)

print(
    complete_accuracy_df.columns.tolist()
)

print(
    "\nFinal complete table:"
)

display(
    complete_accuracy_df[
        display_columns
    ]
)

print("\n" + "=" * 120)
print(
    "FINAL 10-STATION MODELS"
)
print("=" * 120)

final_display_columns = [
    "Model",
    "MAE",
    "RMSE",
    "R2",
    "PredictionAccuracy_±10AQI_%",
    "PredictionAccuracy_±20AQI_%",
    "ValidTestRecords"
]

final_display_columns = [
    column
    for column in final_display_columns
    if column in accuracy_results_10.columns
]

display(
    accuracy_results_10[
        final_display_columns
    ]
)

print("=" * 120)